In [ ]:
# --- Environment setup: run this cell first (Colab or local) -------------------------
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/cto-school/agentic-ai-engineering.git"   # the public course repository
REPO_DIR = Path("/content/agentic-ai-engineering")

if IN_COLAB:
    if not REPO_DIR.exists():
        print("Cloning the course repository ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
        print("Installing requirements (this takes a minute) ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "setup/requirements-core.txt")], check=True)
    os.chdir(REPO_DIR / "day_04_multi_agent_systems")
    # Colab has no .env file. Paste the key you were issued; it is kept only in this runtime.
    from getpass import getpass
    if not os.getenv("OPENROUTER_API_KEY"):
        key = getpass("OPENROUTER_API_KEY (press Enter to stay in mock mode): ").strip()
        if key:
            os.environ["OPENROUTER_API_KEY"] = key
else:
    # Local machine: the key is read from the .env file at the repository root
    # (Day 1.1 explains how to create it from .env.example).
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))

print("Working directory:", os.getcwd())
print("Mode:", "LIVE (OpenRouter key found)" if os.getenv("OPENROUTER_API_KEY") else "MOCK (no key found: deterministic answers, no credit spent)")


# Day 4.1 — A Review Task We Can Measure

## Before you begin

### Learning outcomes

- Read a supplied artifact and describe a defect as a structured, evidenced record.
- Explain why a hidden answer key is what turns "the review looked good" into a measurement.
- Predict two defects yourself before the answer key is revealed.

Architecture reference: [Day 4 diagrams D12](../diagrams/source/day_04.md)

### Expected observation

The artifact prints with line numbers, your prediction comes first, and only then does the answer key reveal that nine defects were seeded.

> **Need an API key?** You do not need one today: every cell runs offline. If you want the optional live experiment, create the `.env` file exactly as shown in **Day 1.1 — Your First Model Call**, at the repository root, then restart the kernel.


## Concept briefing

## Why multiple agents are not the starting point

Adding agents adds model calls, duplicated context, coordination logic, latency, cost and
new failure modes. It is justified only when a task splits into bounded perspectives whose
combined quality beats a simpler system by enough to pay for that complexity.

So today we do not argue about it. We build one general reviewer, one reviewer plus
deterministic tools, and a three-specialist team with a supervisor, run all three over the
same artifact with a hidden answer key, and read the numbers.

We run that comparison twice, against two different reviewers: one with real blind spots,
and one that is already strong. The winner is not the same both times. **The single
reviewer is allowed to win, and in one of the two runs it does.**


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/review_team"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")


## Step 1 — Load the artifact

Everything today reviews the same file. It is deliberately defective classroom code: never copy it into anything real.


In [ ]:
# The artifact under review and the instructor's answer key.
# Both are plain files; nothing here is secret from you, but the answer key is never
# put into a reviewer's prompt.
ARTIFACT_PATH = PROJECT_ROOT / "data" / "seeded_artifact" / "order_service.py"
GOLDEN_PATH   = PROJECT_ROOT / "data" / "golden_defects.json"

SOURCE = ARTIFACT_PATH.read_text(encoding="utf-8")   # the shared, immutable artifact

print("Artifact file :", ARTIFACT_PATH.name)
print("Artifact lines:", len(SOURCE.splitlines()))
print("Answer key    :", GOLDEN_PATH.name)


## Step 2 — Read it the way a reviewer would

Line numbers matter. A finding without a location cannot be checked, merged, or argued with, so every finding we produce today will carry one.


In [ ]:
# Print the artifact with line numbers so we can point at defects precisely.
for number, line in enumerate(SOURCE.splitlines(), 1):
    print(f"{number:>2}: {line}")


## Step 3 — What a finding has to contain

`Finding` is the contract every reviewer and the supervisor agree on. Five of its fields are the claim; two are provenance. If a reviewer cannot fill all of them, it does not have a finding — it has an opinion.


In [ ]:
from review_team import Finding

# Build one finding by hand so the contract is concrete before any reviewer runs.
example = Finding(
    id="EXAMPLE-1",                              # identity of this record
    category="security",                         # correctness | security | maintainability
    line=3,                                      # where in the artifact
    title="Credential-like token is hardcoded",  # the claim, in one line
    evidence=SOURCE.splitlines()[2].strip(),     # the exact source it rests on
    severity="high",                             # low | medium | high | critical
    recommendation="Load secrets from an injected secret store.",
    reviewer="you",                              # who said it
)

for field_name, value in example.as_dict().items():
    print(f"{field_name:>15}: {value}")


### Try it yourself

Scroll back to the printed artifact and pick **two** more defects — one you would call `correctness` and one you would call `maintainability`. Write down the line number and the exact source text for each. Then run the worked solution.


In [ ]:
# --- Worked solution ---
# Two defects you can find by reading alone, written as proper Finding records.
lines = SOURCE.splitlines()

my_findings = [
    Finding(
        id="MINE-1",
        category="correctness",
        line=25,                                  # "return sum(...) / len(items)"
        title="Empty item list causes division by zero",
        evidence=lines[24].strip(),               # index 24 == line 25
        severity="medium",
        recommendation="Decide what an empty order should return before dividing.",
        reviewer="student",
    ),
    Finding(
        id="MINE-2",
        category="maintainability",
        line=26,                                  # "except Exception:"
        title="Broad exception handler hides unrelated failures",
        evidence=lines[25].strip(),
        severity="medium",
        recommendation="Catch only the errors you expect, and log the rest.",
        reviewer="student",
    ),
]

for finding in my_findings:
    print(f"line {finding.line:>3} | {finding.category:<15} | {finding.severity:<6} | "
          f"{finding.title}")
    print(f"         evidence: {finding.evidence}")

print("\nFindings written before seeing the answer key:", len(my_findings))


## Step 4 — Now reveal the answer key

The golden set lists every defect that was deliberately seeded. It exists so we can compute **recall** (how many known defects a system found) and **false positives** (claims that match nothing). It is instructor-owned and never goes into a prompt.


In [ ]:
import json

golden = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))
print("Seeded defects:", len(golden), "\n")

for category in ("correctness", "security", "maintainability"):
    in_category = [item for item in golden if item["category"] == category]
    print(f"{category} ({len(in_category)}):")
    for item in in_category:
        print(f"   line {item['line']:>3}  {item['id']}  [{item['severity']}]  {item['title']}")
    print()


## Step 5 — Score your own review

Same scoring code we will use for every agent today. Notice the rule: a defect can be credited **once**. Reporting it twice is a duplicate, not two discoveries.


In [ ]:
from review_team import ReviewRun, evaluate

# Wrap your two findings in a ReviewRun so the evaluator can score them like any system.
my_run = ReviewRun(system="human_reader", findings=my_findings)
row = evaluate(my_run, GOLDEN_PATH)

print("Known defects  :", row["known_defects"])
print("You found      :", row["found"])
print("Recall         :", row["recall"])
print("False positives:", row["false_positives"])
print("You missed     :", row["missed"])


### Checkpoint

**1. Why must the golden set stay out of the reviewer's prompt?**

<details><summary>Show answer</summary>

Because a reviewer that has been shown the answers is being tested on copying, not on reviewing. Its recall would measure prompt leakage instead of capability, and the comparison between architectures would become meaningless.

</details>

**2. Two reviewers both report the `eval` call on line 20. How many defects were found?**

<details><summary>Show answer</summary>

One. Each known defect is credited once; the second report is counted as a *duplicate*. Counting it twice would let a system inflate its recall simply by repeating itself, which is exactly the failure mode a multi-agent team risks.

</details>

### Recap

- Limitation we saw: "the review looked thorough" is not a measurement — nothing in it can be compared between two systems.
- Layer we added: a structured `Finding` contract plus an instructor-owned golden defect set, scored by `evaluate`.
- Evidence it worked: your two hand-written findings were scored automatically, with recall, false positives and a list of what you missed.
